In [2]:
import sys
from pathlib import Path

import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import timm  

from torch.utils.data import DataLoader

sys.path.append(
    str(
        Path("..")
        .resolve()
        / "src"
    )
)

from dataset_utils import EuroSATDataset
from training_utils import train_model

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cpu


In [5]:
train_df = pd.read_csv(
    "../data/splits/train.csv"
)

validation_df = pd.read_csv(
    "../data/splits/validation.csv"
)

test_df = pd.read_csv(
    "../data/splits/test.csv"
)

with open(
    "../data/splits/metadata.json",
    "r"
) as file:

    metadata = json.load(
        file
    )

MEAN = metadata[
    "band_mean"
]

STD = metadata[
    "band_std"
]

print(
    len(train_df),
    len(validation_df),
    len(test_df)
)

print(
    MEAN
)

print(
    STD
)

16200 5400 5400
[1354.4054651331019, 1118.2439995810425, 1042.9298395302856, 947.626202980324, 1199.4728396116657, 1999.7909091374904, 2369.2229256486303, 2296.8260832308547, 732.0834017831307, 12.113278040002893, 1819.0102785463687, 1118.9239114945024, 2594.1408079849057]
[245.71762907843808, 333.00778264154405, 395.0924913916503, 593.7505558886336, 566.4170016995537, 861.1839900603321, 1086.631390746977, 1117.9817079077502, 404.9197888562917, 4.7758446836604795, 1002.5876831066728, 761.3032349896243, 1231.585810422304]


In [6]:
train_dataset = EuroSATDataset(
    train_df,
    mean=MEAN,
    std=STD,
    image_size=224,
    augment=True
)

validation_dataset = EuroSATDataset(
    validation_df,
    mean=MEAN,
    std=STD,
    image_size=224,
    augment=False
)

test_dataset = EuroSATDataset(
    test_df,
    mean=MEAN,
    std=STD,
    image_size=224,
    augment=False
)

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

Train batches: 2025
Validation batches: 675
Test batches: 675


In [ ]:
import timm

# Create EfficientNetV2 model adapted for 13 input channels and 10 classes
model = timm.create_model(
    'efficientnetv2_rw_m', 
    pretrained=True, 
    in_chans=13, 
    num_classes=10
)

for param in model.parameters():
    param.requires_grad = False

for param in model.get_classifier().parameters():
    param.requires_grad = True

model = model.to(device)

print("Frozen EfficientNetV2 baseline")
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)
print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

In [ ]:
criterion = nn.CrossEntropyLoss()

baseline_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-4
)

baseline_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    baseline_optimizer,
    T_max=10
)

In [ ]:
Path("../models").mkdir(
    parents=True,
    exist_ok=True
)

model, baseline_history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=baseline_optimizer,
    scheduler=baseline_scheduler,
    device=device,
    epochs=10,
    patience=11,
    save_path="../models/efficientnetv2_frozen_best.pth",
    accumulation_steps=1
)

baseline_history_path = Path(
    "../models/efficientnetv2_frozen_history.json"
)
with baseline_history_path.open("w") as file:
    json.dump(baseline_history, file)

print("Frozen baseline training complete.")

In [20]:
model = timm.create_model(
    'efficientnetv2_rw_m',
    pretrained=True,
    in_chans=13,
    num_classes=10
)

# Ensure all model layers are trainable.
for param in model.parameters():
    param.requires_grad = True

model = model.to(device)

print("Fully trainable EfficientNetV2")
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)
print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

Fully trainable EfficientNetV2
Trainable parameters: 51107852
Total parameters: 51107852


In [21]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [22]:
Path("../models").mkdir(
    parents=True,
    exist_ok=True
)

model, history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=10,
    patience=11,
    save_path="../models/efficientnetv2_best.pth",
    accumulation_steps=1
)

finetuning_history_path = Path(
    "../models/efficientnetv2_finetuning_history.json"
)
with finetuning_history_path.open("w") as file:
    json.dump(history, file)

print("Full fine-tuning complete.")

Epoch 01/10 | Train Loss: 0.3598 | Val Loss: 0.1553 | Train Acc: 0.8791 | Val Acc: 0.9519 | Val Macro F1: 0.9501 | 5313.2s
Epoch 02/10 | Train Loss: 0.1563 | Val Loss: 0.3320 | Train Acc: 0.9488 | Val Acc: 0.9570 | Val Macro F1: 0.9552 | 8429.4s
Epoch 03/10 | Train Loss: 0.1192 | Val Loss: 2.2424 | Train Acc: 0.9627 | Val Acc: 0.9580 | Val Macro F1: 0.9564 | 10692.8s
Epoch 04/10 | Train Loss: 0.0925 | Val Loss: 0.1048 | Train Acc: 0.9685 | Val Acc: 0.9731 | Val Macro F1: 0.9726 | 10710.6s
Epoch 05/10 | Train Loss: 0.0703 | Val Loss: 0.2391 | Train Acc: 0.9759 | Val Acc: 0.9557 | Val Macro F1: 0.9542 | 10696.3s
Epoch 06/10 | Train Loss: 0.0461 | Val Loss: 0.0594 | Train Acc: 0.9844 | Val Acc: 0.9830 | Val Macro F1: 0.9826 | 10668.4s
Epoch 07/10 | Train Loss: 0.0356 | Val Loss: 0.0661 | Train Acc: 0.9891 | Val Acc: 0.9809 | Val Macro F1: 0.9804 | 10633.4s
Epoch 08/10 | Train Loss: 0.0219 | Val Loss: 0.0602 | Train Acc: 0.9932 | Val Acc: 0.9839 | Val Macro F1: 0.9834 | 10620.8s
Epoch 09/1

In [28]:
baseline_history_path = Path(
    "../models/efficientnetv2_frozen_history.json"
)
finetuning_history_path = Path(
    "../models/efficientnetv2_finetuning_history.json"
)

if not isinstance(globals().get("baseline_history"), dict):
    if not baseline_history_path.is_file():
        raise RuntimeError(
            "Frozen baseline history is missing. Run the frozen baseline "
            "training cell before plotting."
        )
    with baseline_history_path.open() as file:
        baseline_history = json.load(file)
else:
    with baseline_history_path.open("w") as file:
        json.dump(baseline_history, file)

if not isinstance(globals().get("history"), dict):
    if not finetuning_history_path.is_file():
        raise RuntimeError(
            "Fine-tuning history is missing. Run the fine-tuning training "
            "cell before plotting."
        )
    with finetuning_history_path.open() as file:
        history = json.load(file)
else:
    with finetuning_history_path.open("w") as file:
        json.dump(history, file)

baseline_epochs = range(
    1,
    len(baseline_history["validation_accuracy"]) + 1
)

finetuning_epochs = range(
    1,
    len(history["validation_accuracy"]) + 1
)

plt.figure(figsize=(8, 5))
plt.plot(
    baseline_epochs,
    baseline_history["validation_accuracy"],
    marker="o",
    label="Frozen Baseline"
)
plt.plot(
    finetuning_epochs,
    history["validation_accuracy"],
    marker="o",
    label="Full Fine-Tuning"
)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("EfficientNetV2: Validation Accuracy Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
Path("../figures").mkdir(parents=True, exist_ok=True)
plt.savefig(
    "../figures/efficientnetv2_validation_accuracy_comparison.png",
    dpi=300
)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(baseline_history["validation_macro_f1"]) + 1),
    baseline_history["validation_macro_f1"],
    marker="o",
    label="Frozen Baseline"
)
plt.plot(
    range(1, len(history["validation_macro_f1"]) + 1),
    history["validation_macro_f1"],
    marker="o",
    label="Full Fine-Tuning"
)
plt.xlabel("Epoch")
plt.ylabel("Validation Macro F1")
plt.title("EfficientNetV2: Validation Macro F1 Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    "../figures/efficientnetv2_validation_macro_f1_comparison.png",
    dpi=300
)
plt.show()

RuntimeError: Frozen baseline history is missing. Run the frozen baseline training cell before plotting.

In [ ]:
experiments = {
    "Frozen Baseline": baseline_history,
    "Full Fine-Tuning": history
}

for name, results in experiments.items():

    best_accuracy = max(
        results["validation_accuracy"]
    )

    best_macro_f1 = max(
        results["validation_macro_f1"]
    )

    best_loss = min(
        results["validation_loss"]
    )

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Best validation accuracy: {best_accuracy:.4f}")
    print(f"Best validation macro F1: {best_macro_f1:.4f}")
    print(f"Lowest validation loss:   {best_loss:.4f}")